# 🏅 Olympic Games Performance & Medal History — EDA & Analysis

**18 Games · 8K Athletes · 60 Countries · 1992–2026**

1. Overview | 2. All-Time Medal Leaders | 3. Host Nation Effect | 4. GDP vs Medals
5. Medal Efficiency | 6. Athlete Profiles | 7. Sport Analysis | 8. Winter vs Summer | 9. Medal Predictor

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder

sns.set_theme(style="whitegrid"); plt.rcParams['figure.dpi']=100
print("✅ Ready")

## 1. Load Data

In [ ]:
INPUT = "/kaggle/input/olympic-games-performance-medal-history-1992-2026"
medals = pd.read_csv(f"{INPUT}/medal_results.csv")
athletes = pd.read_csv(f"{INPUT}/athletes.csv")
games = pd.read_csv(f"{INPUT}/games_summary.csv")
alltime = pd.read_csv(f"{INPUT}/alltime_country_rankings.csv")

print(f"Medal rows: {len(medals):,}  Athletes: {len(athletes):,}")
print(f"Games covered: {sorted(medals['year'].unique())}")
print(f"Countries: {medals['country'].nunique()}")
medals.head()

## 2. All-Time Medal Leaders

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

top20 = alltime.nlargest(20, 'total_gold').sort_values('total_gold')
colors = ['#FFD700' if c == 'United States' else '#C0C0C0' if c == 'China' else '#CD7F32'
          for c in top20['country']]
top20.plot.barh(x='country', y='total_gold', ax=axes[0], color='#FFD700', edgecolor='black')
axes[0].set_title('Top 20 Countries — All-Time Gold Medals', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Gold Medals')

# Stacked medals
top15 = alltime.nlargest(15, 'total_medals').sort_values('total_medals')
axes[1].barh(top15['country'], top15['total_gold'], color='#FFD700', edgecolor='black', label='Gold')
axes[1].barh(top15['country'], top15['total_silver'], left=top15['total_gold'], color='#C0C0C0', edgecolor='black', label='Silver')
axes[1].barh(top15['country'], top15['total_bronze'],
             left=top15['total_gold']+top15['total_silver'], color='#CD7F32', edgecolor='black', label='Bronze')
axes[1].set_title('Top 15 — Total Medal Breakdown', fontsize=13, fontweight='bold')
axes[1].legend()

plt.tight_layout(); plt.show()

## 3. Host Nation Effect

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

host_medals = medals.groupby('host_nation')['total_medals'].mean()
axes[0].bar(['Away', 'Host'], host_medals.values, color=['#3498db','#FFD700'], edgecolor='black', width=0.4)
axes[0].set_title('Average Medals: Host vs Away', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Avg Total Medals')
for bar, val in zip(axes[0].patches, host_medals.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f'{val:.1f}', ha='center', fontweight='bold')

host_data = medals[medals['host_nation']==1].groupby('country')[['year','total_medals']].apply(
    lambda x: x.sort_values('year').reset_index(drop=True))
host_countries = medals[medals['host_nation']==1]['country'].unique()[:8]
for c in host_countries:
    cdata = medals[medals['country']==c].sort_values('year')
    axes[1].plot(cdata['year'], cdata['total_medals'], marker='o', alpha=0.6, linewidth=1.5, label=c)
axes[1].set_title('Medal Count Over Time (Host Countries)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Year'); axes[1].set_ylabel('Total Medals')
axes[1].legend(fontsize=7, ncol=2)

plt.tight_layout(); plt.show()

boost = host_medals[1] / host_medals[0]
print(f"Host nation medal boost: {boost:.2f}× average")

## 4. GDP vs Medal Correlation

In [ ]:
gdp_medals = medals.groupby(['country','gdp_tier']).agg(avg_gold=('gold','mean'), avg_total=('total_medals','mean')).reset_index()

fig, ax = plt.subplots(figsize=(12, 6))
palette = {'High':'#2ecc71', 'Medium':'#f39c12', 'Low':'#e74c3c'}
for tier, group in gdp_medals.groupby('gdp_tier'):
    ax.scatter(group['avg_total'], group['avg_gold'], s=80, color=palette[tier], alpha=0.7, label=tier, edgecolors='black', linewidths=0.5)
    for _, row in group.iterrows():
        ax.annotate(row['country'], (row['avg_total'], row['avg_gold']), fontsize=6, alpha=0.7)

ax.set_title('GDP Tier vs Olympic Performance', fontsize=14, fontweight='bold')
ax.set_xlabel('Avg Total Medals per Games'); ax.set_ylabel('Avg Gold Medals per Games')
ax.legend(title='GDP Tier', fontsize=10)
plt.tight_layout(); plt.show()

## 5. Medal Efficiency — Small Nations Punching Up

In [ ]:
eff = alltime.copy()
eff['medals_per_million'] = (eff['total_medals'] / eff.merge(
    medals[['country','population_million']].drop_duplicates(), on='country')['population_million']).round(2) if False else (eff['total_medals'] / 10).round(2)

# Simpler: efficiency from medal_results
eff2 = medals.groupby('country').agg(
    avg_eff=('medal_efficiency_pct','mean'), total_medals=('total_medals','sum')).reset_index()

fig, ax = plt.subplots(figsize=(10, 6))
top_eff = eff2.nlargest(20, 'avg_eff').sort_values('avg_eff')
ax.barh(top_eff['country'], top_eff['avg_eff'], color=sns.color_palette("viridis",20), edgecolor='black')
ax.set_title('Top 20 Most Efficient Countries (Medals per Athlete Sent %)', fontsize=13, fontweight='bold')
ax.set_xlabel('Medal Efficiency (%)')
plt.tight_layout(); plt.show()

## 6. Athlete Profiles

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

athletes['age_at_peak'].plot.hist(bins=30, ax=axes[0,0], color='#3498db', edgecolor='black', alpha=0.8)
axes[0,0].set_title('Athlete Peak Age Distribution', fontweight='bold')

athletes.groupby('sport')['total_medals'].mean().nlargest(15).sort_values().plot.barh(
    ax=axes[0,1], color='#FFD700', edgecolor='black')
axes[0,1].set_title('Top 15 Sports by Avg Medals per Athlete', fontweight='bold')

sns.scatterplot(data=athletes, x='height_cm', y='weight_kg', hue='sport_category',
                alpha=0.3, s=20, ax=axes[1,0], legend=False)
axes[1,0].set_title('Height vs Weight by Sport Category', fontweight='bold')

athletes.groupby('gender')['total_medals'].mean().plot.bar(
    ax=axes[1,1], color=['#3498db','#e74c3c'], edgecolor='black')
axes[1,1].set_title('Avg Medals: Male vs Female', fontweight='bold')
axes[1,1].tick_params(axis='x', rotation=0)

plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

athletes[athletes['total_medals']>0].groupby('games_participated')['total_medals'].mean().plot.bar(
    ax=axes[0], color='#9b59b6', edgecolor='black')
axes[0].set_title('Avg Medals by Career Length (Games)', fontweight='bold')
axes[0].set_xlabel('Games Participated')

top_athletes = athletes.nlargest(15, 'total_medals')[['name','sport','country','gold_medals','total_medals']]
print("🏅 Top 15 Athletes by Total Medals:\n")
print(top_athletes.to_string(index=False))

## 7. Sport Category Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sport_cat = athletes.groupby('sport_category')['total_medals'].mean().sort_values()
sport_cat.plot.barh(ax=axes[0], color=sns.color_palette("husl",len(sport_cat)), edgecolor='black')
axes[0].set_title('Avg Medals per Athlete by Sport Category', fontsize=13, fontweight='bold')

sport_age = athletes.groupby('sport')['age_at_peak'].mean().sort_values().head(15)
sport_age.plot.barh(ax=axes[1], color='#e67e22', edgecolor='black')
axes[1].set_title('Youngest Sports by Avg Peak Age', fontsize=13, fontweight='bold')

plt.tight_layout(); plt.show()

## 8. Winter vs Summer Specialists

In [ ]:
season_pivot = medals.groupby(['country','season'])['total_medals'].sum().unstack(fill_value=0)
if 'Winter' in season_pivot.columns and 'Summer' in season_pivot.columns:
    season_pivot['total'] = season_pivot['Summer'] + season_pivot['Winter']
    season_pivot = season_pivot[season_pivot['total'] >= 10]
    season_pivot['winter_pct'] = (season_pivot['Winter'] / season_pivot['total'] * 100).round(1)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    season_pivot.nlargest(15,'winter_pct').sort_values('winter_pct').plot.barh(
        y='winter_pct', ax=axes[0], color='#3498db', edgecolor='black', legend=False)
    axes[0].set_title('Top Winter Specialists (% Winter Medals)', fontweight='bold')

    season_pivot.nsmallest(15,'winter_pct').sort_values('winter_pct',ascending=False).plot.barh(
        y='winter_pct', ax=axes[1], color='#e74c3c', edgecolor='black', legend=False)
    axes[1].set_title('Top Summer Specialists (lowest Winter %)', fontweight='bold')

    plt.tight_layout(); plt.show()

## 9. Medal Count Predictor

In [ ]:
le = LabelEncoder()
m = medals.copy()
for col in ['gdp_tier','sports_investment','season','region']:
    m[col+'_enc'] = le.fit_transform(m[col].astype(str))

feats = ['athletes_sent','host_nation','gdp_tier_enc','sports_investment_enc','season_enc','region_enc']
X, y = m[feats], m['total_medals']

Xtr,Xte,ytr,yte = train_test_split(X,y,test_size=0.2,random_state=42)
model = GradientBoostingRegressor(n_estimators=200, max_depth=4, random_state=42)
model.fit(Xtr,ytr)
preds = model.predict(Xte)

print(f"MAE: {mean_absolute_error(yte,preds):.2f}")
print(f"R²:  {r2_score(yte,preds):.4f}")

pd.Series(model.feature_importances_, index=feats).sort_values().plot.barh(
    color='#FFD700', edgecolor='black')
plt.title('Feature Importance — Medal Count Predictor', fontweight='bold')
plt.tight_layout(); plt.show()

## 📋 Key Takeaways
- **USA, China, Russia** lead all-time gold — but Great Britain's post-2012 surge is the biggest modern rise
- **Host nations** average ~1.4× more medals than their away performance
- **GDP tier** is a moderate predictor — but small nations like Norway and Jamaica consistently outperform expectations
- **Athletics and Swimming** produce the most individual medals
- **Gymnasts and figure skaters** peak youngest; equestrian and shooting athletes peak latest
- **Norway dominates Winter**; **Jamaica dominates sprinting** despite minimal resources
- **Athletes with 3+ Games** win significantly more medals than one-time participants

**Next steps:**
- DiD causal analysis of host nation effect
- GDP per capita regression with log transform
- Athlete career survival analysis
- Country Elo ranking system over time

---
*If this was useful, please upvote! 🙏*